In [ ]:
"""
================================================================================
COMPARATIVE ANALYSIS: CRYPTOGUARD-MLP vs LINEAR REGRESSION vs LOGISTIC REGRESSION vs SVM vs XGBoost vs LightGBM vs CatBoost
5-Fold Stratified CV with proper train/val/test split (14.4k/1.6k/4k per fold)
Early Stopping on Validation Data for All Models
================================================================================
"""

# ============ SYSTEM & UTILITIES ============
import os
import sys
import gc
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ============ DEEP LEARNING (MLP) ============
import tensorflow as tf
from tensorflow import keras
from keras import Sequential, regularizers
from keras.layers import Dense, Dropout, Input, BatchNormalization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ============ SKLEARN - DATA & METRICS ============
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (matthews_corrcoef, accuracy_score, precision_score, 
                             recall_score, f1_score, confusion_matrix)

# ============ RANDOM SEED & ENVIRONMENT ============
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import random
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.set_visible_devices([], 'GPU')

# ============ CONFIG ============
DATASET = "D"
DATE = "13082026"
CV_FOLDS = 5
EPOCHS_FINAL = 100
PATIENCE_ES = 5
PATIENCE_LR = 5

# ============ MODEL SELECTION ============
# Choose which models to run: 'MLP', 'LinReg', 'LogReg', 'SVM', 'XGBoost', 'LightGBM', 'CatBoost'
# Order: MLP → Linear Regression → Logistic Regression → SVM → XGBoost → LightGBM → CatBoost
MODELS_TO_RUN = ['MLP', 'LinReg', 'LogReg', 'SVM', 'XGBoost', 'LightGBM', 'CatBoost']

# ============ INPUT FILE (Part 1 Results - FIXED) ============
FILE_RESULTS = "./0. D_CryptoGuard-MLP_GridSearchCV_5Fold_08082026/1.1_D_CryptoGuard-MLP_GridSearchCV_5Fold_Results_08082026.csv"

# ============ OUTPUT DIRECTORY & FILES ============
OUT_DIR = f"./1. CryptoGuard-MLP_ComparativeAnalysis_{DATE}"
os.makedirs(OUT_DIR, exist_ok=True)

FILE_METRICS_MASTER = f"{OUT_DIR}/1.1_CryptoGuard-MLP_ComparativeAnalysis_MasterResults_{DATE}.csv"
FILE_METRICS_AGGS = f"{OUT_DIR}/1.2_CryptoGuard-MLP_ComparativeAnalysis_AggregateResults_{DATE}.csv"

# ============ HELPER FUNCTIONS ============
def print_header(title):
    print("\n" + "="*80)
    print(title)
    print("="*80)

def print_step(step_num, title):
    print(f"\n[STEP {step_num}] {title}")
    print("-" * 80)

def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    
    return {
        'TP': int(tp), 
        'FP': int(fp), 
        'FN': int(fn), 
        'TN': int(tn),
        'Accuracy': f"{acc*100:.2f}%", 
        'Precision': f"{prec*100:.2f}%", 
        'Recall': f"{rec*100:.2f}%", 
        'F-measure': f"{f1*100:.2f}%",
        'MCC': f"{mcc:.4f}",
        'FPR': f"{fpr*100:.2f}%", 
        'FNR': f"{fnr*100:.2f}%"
    }

def create_mlp_model(hidden_layer1, hidden_layer2, hidden_layer3, 
                     dropout_rate, l2_reg, learning_rate, input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(hidden_layer1, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(hidden_layer2, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(hidden_layer3, activation='relu', kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid', kernel_regularizer=regularizers.l2(l2_reg))
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def create_linreg_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(1, activation='linear')
    ])
    model.compile(
        loss='mean_squared_error',
        optimizer=keras.optimizers.SGD(learning_rate=learning_rate),
        metrics=['accuracy']
    )
    return model

def create_logreg_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(1, activation='sigmoid', 
              kernel_regularizer=regularizers.l2(0.1))
    ])
    model.compile(
        loss='binary_crossentropy',
        optimizer=keras.optimizers.SGD(learning_rate=learning_rate),
        metrics=['accuracy']
    )
    return model

def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES, 
                     restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, 
                         patience=PATIENCE_LR, min_lr=1e-6, verbose=0)
    ]

# ============ MAIN EXECUTION ============
if __name__ == "__main__":
    print_header("COMPARATIVE ANALYSIS: CRYPTOGUARD-MLP vs 6 BASELINE MODELS")

    # ============ STEP 1: LOAD PART 1 GRIDSEARCH RESULTS ============
    print_step(1, "LOADING PART 1 GRIDSEARCH RESULTS")
    
    df_results = pd.read_csv(FILE_RESULTS)
    best_row = df_results.iloc[0]
    
    h1 = int(best_row['param_hidden_layer1'])
    h2 = int(best_row['param_hidden_layer2'])
    h3 = int(best_row['param_hidden_layer3'])
    dr = float(best_row['param_dropout_rate'])
    l2r = float(best_row['param_l2_reg'])
    lr = float(best_row['param_learning_rate'])
    bs = int(best_row['param_batch_size'])
    
    cv_train_mcc = float(best_row['mean_train_score'])
    cv_val_mcc = float(best_row['mean_test_score'])
    cv_gap_mcc = float(best_row['Gap_MCC'])
    rank = int(best_row['rank_test_score'])
    
    print(f"\n✓ Strategy 1: Best MCC (Rank {rank})")
    print(f"  Architecture      : {h1}-{h2}-{h3}")
    print(f"  Dropout Rate      : {dr}")
    print(f"  L2 Regularization : {l2r}")
    print(f"  Learning Rate     : {lr}")
    print(f"  Batch Size        : {bs}")
    print(f"  Mean Train Score  : {cv_train_mcc:.4f}")
    print(f"  Mean Test Score   : {cv_val_mcc:.4f}")
    print(f"  Gap MCC           : {cv_gap_mcc:.4f}")

    # ============ STEP 2: LOAD FULL DATASET ============
    print_step(2, "LOADING FULL DATASET")
    
    df = pd.read_excel(f'./Datasets/{DATASET}.xlsx')
    X = df.drop(columns=['CLASS_LABEL'])
    y = df['CLASS_LABEL']
    input_dim = X.shape[1]
    
    print(f"\n✓ X shape: {X.shape}")
    print(f"✓ y shape: {y.shape}")
    print(f"✓ Input dimension: {input_dim}")
    print(f"✓ Class distribution: {y.value_counts().to_dict()}")

    # ============ STEP 3: 5-FOLD STRATIFIED CV (FOLD BY FOLD) ============
    print_step(3, "5-FOLD STRATIFIED CV (MLP → LINEAR REGRESSION → LOGISTIC REGRESSION → SVM → XGBoost → LightGBM → CatBoost)")
    
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    fold_metrics_list = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n[Fold {fold}/{CV_FOLDS}]")
        
        # ========== Split 1: Outer fold (train 80%, test 20%) ==========
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx].values.astype(np.float32)
        y_test = y.iloc[test_idx].values.astype(np.float32)
        
        # ========== Split 2: Inner stratified split (10% validation from training) ==========
        X_train_real, X_val, y_train_real, y_val = train_test_split(
            X_train, y_train,
            test_size=0.10,
            random_state=SEED,
            stratify=y_train
        )
        
        # ========== Scale (fit on actual training, transform val and test) ==========
        scaler = StandardScaler()
        X_train_real_sc = scaler.fit_transform(X_train_real).astype(np.float32)
        X_val_sc = scaler.transform(X_val).astype(np.float32)
        X_test_sc = scaler.transform(X_test).astype(np.float32)
        
        # ========== MODEL 1: CRYPTOGUARD-MLP (PART 2A SETUP) ==========
        if 'MLP' in MODELS_TO_RUN:
            print(f"  Training CryptoGuard-MLP...", end='', flush=True)
            model_mlp = create_mlp_model(h1, h2, h3, dr, l2r, lr, input_dim)
            history_mlp = model_mlp.fit(
                X_train_real_sc, y_train_real,
                epochs=EPOCHS_FINAL,
                batch_size=bs,
                validation_data=(X_val_sc, y_val),
                callbacks=get_callbacks(),
                verbose=0
            )
            epochs_trained_mlp = len(history_mlp.history['loss'])
            y_pred_mlp = (model_mlp.predict(X_test_sc, verbose=0, batch_size=bs) > 0.5).astype(int).flatten()
            metrics_mlp = calculate_metrics(y_test, y_pred_mlp)
            metrics_mlp['Epochs_Trained'] = epochs_trained_mlp
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'CryptoGuard-MLP', **metrics_mlp})
            print(f" ✓ Acc={metrics_mlp['Accuracy']}, MCC={metrics_mlp['MCC']}, Epochs={epochs_trained_mlp}")
            del model_mlp, history_mlp
            tf.keras.backend.clear_session()
            gc.collect()
        
        # ========== MODEL 2: LINEAR REGRESSION ==========
        if 'LinReg' in MODELS_TO_RUN:
            print(f"  Training Linear Regression...", end='', flush=True)
            model_linreg = create_linreg_model(input_dim)
            history_linreg = model_linreg.fit(
                X_train_real_sc, y_train_real,
                epochs=EPOCHS_FINAL,
                batch_size=bs,
                validation_data=(X_val_sc, y_val),
                callbacks=get_callbacks(),
                verbose=0
            )
            epochs_trained_linreg = len(history_linreg.history['loss'])
            y_pred_linreg = np.where(model_linreg.predict(X_test_sc, verbose=0) > 0.5, 1, 0).flatten()
            metrics_linreg = calculate_metrics(y_test, y_pred_linreg)
            metrics_linreg['Epochs_Trained'] = epochs_trained_linreg
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'Linear Regression', **metrics_linreg})
            print(f" ✓ Acc={metrics_linreg['Accuracy']}, MCC={metrics_linreg['MCC']}, Epochs={epochs_trained_linreg}")
            del model_linreg, history_linreg
            tf.keras.backend.clear_session()
            gc.collect()
        
        # ========== MODEL 3: LOGISTIC REGRESSION ==========
        if 'LogReg' in MODELS_TO_RUN:
            print(f"  Training Logistic Regression...", end='', flush=True)
            model_logreg = create_logreg_model(input_dim)
            history_logreg = model_logreg.fit(
                X_train_real_sc, y_train_real,
                epochs=EPOCHS_FINAL,
                batch_size=bs,
                validation_data=(X_val_sc, y_val),
                callbacks=get_callbacks(),
                verbose=0
            )
            epochs_trained_logreg = len(history_logreg.history['loss'])
            y_pred_logreg = (model_logreg.predict(X_test_sc, verbose=0, batch_size=bs) > 0.5).astype(int).flatten()
            metrics_logreg = calculate_metrics(y_test, y_pred_logreg)
            metrics_logreg['Epochs_Trained'] = epochs_trained_logreg
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'Logistic Regression', **metrics_logreg})
            print(f" ✓ Acc={metrics_logreg['Accuracy']}, MCC={metrics_logreg['MCC']}, Epochs={epochs_trained_logreg}")
            del model_logreg, history_logreg
            tf.keras.backend.clear_session()
            gc.collect()
        
        # ========== MODEL 4: SVM ==========
        if 'SVM' in MODELS_TO_RUN:
            print(f"  Training SVM...", end='', flush=True)
            svm_model = SVC(kernel='rbf', C=1, random_state=SEED)
            svm_model.fit(X_train_real_sc, y_train_real)
            y_pred_svm = svm_model.predict(X_test_sc)
            metrics_svm = calculate_metrics(y_test, y_pred_svm)
            metrics_svm['Epochs_Trained'] = '-'
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'SVM', **metrics_svm})
            print(f" ✓ Acc={metrics_svm['Accuracy']}, MCC={metrics_svm['MCC']}")
            del svm_model
            gc.collect()
        
        # ========== MODEL 5: XGBOOST (WITH EARLY STOPPING) ==========
        if 'XGBoost' in MODELS_TO_RUN:
            print(f"  Training XGBoost...", end='', flush=True)
            xgb_model = XGBClassifier(
                n_estimators=100, learning_rate=1,
                max_depth=6, eval_metric='logloss',
                early_stopping_rounds=PATIENCE_ES,
                random_state=SEED, verbosity=0)
            xgb_model.fit(
                X_train_real_sc, y_train_real,
                eval_set=[(X_val_sc, y_val)],
                verbose=False)
            y_pred_xgb = xgb_model.predict(X_test_sc)
            metrics_xgb = calculate_metrics(y_test, y_pred_xgb)
            metrics_xgb['Epochs_Trained'] = '-'
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'XGBoost', **metrics_xgb})
            print(f" ✓ Acc={metrics_xgb['Accuracy']}, MCC={metrics_xgb['MCC']}")
            del xgb_model
            gc.collect()
        
        # ========== MODEL 6: LIGHTGBM (WITH EARLY STOPPING) ==========
        if 'LightGBM' in MODELS_TO_RUN:
            print(f"  Training LightGBM...", end='', flush=True)
            lgb_model = LGBMClassifier(
                n_estimators=100, learning_rate=1,
                max_depth=6, random_state=SEED, verbosity=0)
            lgb_model.fit(
                X_train_real_sc, y_train_real,
                eval_set=[(X_val_sc, y_val)],
                callbacks=[lgb.early_stopping(PATIENCE_ES)])
            y_pred_lgb = lgb_model.predict(X_test_sc)
            metrics_lgb = calculate_metrics(y_test, y_pred_lgb)
            metrics_lgb['Epochs_Trained'] = '-'
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'LightGBM', **metrics_lgb})
            print(f" ✓ Acc={metrics_lgb['Accuracy']}, MCC={metrics_lgb['MCC']}")
            del lgb_model
            gc.collect()
        
        # ========== MODEL 7: CATBOOST (WITH EARLY STOPPING) ==========
        if 'CatBoost' in MODELS_TO_RUN:
            print(f"  Training CatBoost...", end='', flush=True)
            cat_model = CatBoostClassifier(
                iterations=100, learning_rate=1,
                depth=6, early_stopping_rounds=PATIENCE_ES,
                random_state=SEED, verbose=0)
            cat_model.fit(
                X_train_real_sc, y_train_real,
                eval_set=(X_val_sc, y_val))
            y_pred_cat = cat_model.predict(X_test_sc)
            metrics_cat = calculate_metrics(y_test, y_pred_cat)
            metrics_cat['Epochs_Trained'] = '-'
            fold_metrics_list.append({'Fold': f'Fold {fold}', 'Model': 'CatBoost', **metrics_cat})
            print(f" ✓ Acc={metrics_cat['Accuracy']}, MCC={metrics_cat['MCC']}")
            del cat_model
            gc.collect()
        
        # Clean up
        del X_train, X_test, X_train_real, X_val, X_train_real_sc, X_val_sc, X_test_sc
        gc.collect()

    # ============ STEP 4: SAVE RESULTS ============
    print_step(4, "SAVING CSV RESULTS")
    
    fold_metrics_df = pd.DataFrame(fold_metrics_list)
    
    # ========== MASTERRESULTS (All folds + Mean per model) ==========
    # Order: MLP → Linear Regression → Logistic Regression → SVM → XGBoost → LightGBM → CatBoost
    master_rows = []
    
    for model_name in ['CryptoGuard-MLP', 'Linear Regression', 'Logistic Regression', 'SVM', 'XGBoost', 'LightGBM', 'CatBoost']:
        if model_name == 'CryptoGuard-MLP' and 'MLP' not in MODELS_TO_RUN:
            continue
        if model_name == 'Linear Regression' and 'LinReg' not in MODELS_TO_RUN:
            continue
        if model_name == 'Logistic Regression' and 'LogReg' not in MODELS_TO_RUN:
            continue
        if model_name == 'SVM' and 'SVM' not in MODELS_TO_RUN:
            continue
        if model_name == 'XGBoost' and 'XGBoost' not in MODELS_TO_RUN:
            continue
        if model_name == 'LightGBM' and 'LightGBM' not in MODELS_TO_RUN:
            continue
        if model_name == 'CatBoost' and 'CatBoost' not in MODELS_TO_RUN:
            continue
        
        # Get all folds for this model (5 rows)
        model_folds = fold_metrics_df[fold_metrics_df['Model'] == model_name]
        
        # Add 5 fold rows
        for idx, row in model_folds.iterrows():
            master_rows.append(row.to_dict())
        
        # Calculate and add Mean row (NO TP/FP/FN/TN/Epochs_Trained)
        accuracy_vals = [float(x.strip('%')) for x in model_folds['Accuracy']]
        fpr_vals = [float(x.strip('%')) for x in model_folds['FPR']]
        fnr_vals = [float(x.strip('%')) for x in model_folds['FNR']]
        mcc_vals = [float(x) for x in model_folds['MCC']]
        precision_vals = [float(x.strip('%')) for x in model_folds['Precision']]
        recall_vals = [float(x.strip('%')) for x in model_folds['Recall']]
        f_measure_vals = [float(x.strip('%')) for x in model_folds['F-measure']]
        
        mean_row = {
            'Model': model_name,
            'Fold': 'Mean',
            'Epochs_Trained': '-',
            'TP': '-',
            'FP': '-',
            'FN': '-',
            'TN': '-',
            'Accuracy': f"{np.mean(accuracy_vals):.2f}±{np.std(accuracy_vals):.2f}%",
            'FPR': f"{np.mean(fpr_vals):.2f}±{np.std(fpr_vals):.2f}%",
            'FNR': f"{np.mean(fnr_vals):.2f}±{np.std(fnr_vals):.2f}%",
            'MCC': f"{np.mean(mcc_vals):.4f}±{np.std(mcc_vals):.4f}",
            'Precision': f"{np.mean(precision_vals):.2f}±{np.std(precision_vals):.2f}%",
            'Recall': f"{np.mean(recall_vals):.2f}±{np.std(recall_vals):.2f}%",
            'F-measure': f"{np.mean(f_measure_vals):.2f}±{np.std(f_measure_vals):.2f}%"
        }
        master_rows.append(mean_row)
    
    master_df = pd.DataFrame(master_rows)
    
    # Reorder columns to exact order requested
    ordered_cols = ['Model', 'Fold', 'Epochs_Trained', 'TP', 'FP', 'FN', 'TN', 'Accuracy', 'FPR', 'FNR', 'MCC', 'Precision', 'Recall', 'F-measure']
    master_df = master_df[ordered_cols]
    
    master_df.to_csv(FILE_METRICS_MASTER, index=False)
    print(f"\n✓ Saved MasterResults: {FILE_METRICS_MASTER}")
    
    # ========== AGGREGATERESULTS (Summary by model - NO EPOCHS_TRAINED) ==========
    # Order: MLP → Linear Regression → Logistic Regression → SVM → XGBoost → LightGBM → CatBoost
    agg_rows = []
    for model_name in ['CryptoGuard-MLP', 'Linear Regression', 'Logistic Regression', 'SVM', 'XGBoost', 'LightGBM', 'CatBoost']:
        if model_name == 'CryptoGuard-MLP' and 'MLP' not in MODELS_TO_RUN:
            continue
        if model_name == 'Linear Regression' and 'LinReg' not in MODELS_TO_RUN:
            continue
        if model_name == 'Logistic Regression' and 'LogReg' not in MODELS_TO_RUN:
            continue
        if model_name == 'SVM' and 'SVM' not in MODELS_TO_RUN:
            continue
        if model_name == 'XGBoost' and 'XGBoost' not in MODELS_TO_RUN:
            continue
        if model_name == 'LightGBM' and 'LightGBM' not in MODELS_TO_RUN:
            continue
        if model_name == 'CatBoost' and 'CatBoost' not in MODELS_TO_RUN:
            continue
            
        model_data = fold_metrics_df[fold_metrics_df['Model'] == model_name]
        
        accuracy_vals = [float(x.strip('%')) for x in model_data['Accuracy']]
        fpr_vals = [float(x.strip('%')) for x in model_data['FPR']]
        fnr_vals = [float(x.strip('%')) for x in model_data['FNR']]
        mcc_vals = [float(x) for x in model_data['MCC']]
        precision_vals = [float(x.strip('%')) for x in model_data['Precision']]
        recall_vals = [float(x.strip('%')) for x in model_data['Recall']]
        f_measure_vals = [float(x.strip('%')) for x in model_data['F-measure']]
       
        agg_row = {
            'Model': model_name,
            'Accuracy': f"{np.mean(accuracy_vals):.2f}±{np.std(accuracy_vals):.2f}%",
            'FPR': f"{np.mean(fpr_vals):.2f}±{np.std(fpr_vals):.2f}%",
            'FNR': f"{np.mean(fnr_vals):.2f}±{np.std(fnr_vals):.2f}%",
            'MCC': f"{np.mean(mcc_vals):.4f}±{np.std(mcc_vals):.4f}",
            'Precision': f"{np.mean(precision_vals):.2f}±{np.std(precision_vals):.2f}%",
            'Recall': f"{np.mean(recall_vals):.2f}±{np.std(recall_vals):.2f}%",
            'F-measure': f"{np.mean(f_measure_vals):.2f}±{np.std(f_measure_vals):.2f}%",
        }
        agg_rows.append(agg_row)
    
    agg_df = pd.DataFrame(agg_rows)
    
    # Reorder columns to exact order
    agg_cols_ordered = ['Model', 'Accuracy', 'FPR', 'FNR', 'MCC', 'Precision', 'Recall', 'F-measure']
    agg_df = agg_df[agg_cols_ordered]
    
    agg_df.to_csv(FILE_METRICS_AGGS, index=False)
    print(f"✓ Saved AggregateResults: {FILE_METRICS_AGGS}")

    # ========== SUMMARY ==========
    print("\n" + "="*80)
    print("COMPARATIVE ANALYSIS COMPLETE!")
    print("="*80)
    
    print(f"\nModels Executed: {', '.join(MODELS_TO_RUN)}")
    print(f"\nOutput Files:")
    print(f"  ✓ {FILE_METRICS_MASTER}")
    print(f"  ✓ {FILE_METRICS_AGGS}")
    print(f"\nOutput Directory: {OUT_DIR}")
    print("="*80)

In [ ]:
"""
================================================================================
BAR CHART DATA PREPARATION
Extract mean ± std from Aggregate Results and create 2-sheet Excel file
CryptoGuard-MLP at Last Row
================================================================================
"""

import pandas as pd
import re

# ============ READ AGGREGATE RESULTS ============
df = pd.read_csv("./1. CryptoGuard-MLP_ComparativeAnalysis_13082026/1.2_CryptoGuard-MLP_ComparativeAnalysis_AggregateResults_13082026.csv")

print("✓ Loaded AggregateResults CSV")
print(f"✓ Models: {', '.join(df['Model'].tolist())}")

# ============ FUNCTION TO EXTRACT MEAN & STD ============
def extract_mean_std(value):
    """Extract mean, std (including negative values), and check if value has % symbol"""
    match = re.search(r'(-?[\d.]+)±(-?[\d.]+)', str(value))
    if match:
        mean = float(match.group(1))
        std = float(match.group(2))
        is_percentage = '%' in str(value)  # Check if % symbol exists
        return mean, std, is_percentage
    return None, None, None

# ============ SHEET 1: ACCURACY, FPR, FNR, MCC ============
print("\n[Sheet 1] Creating Chart_1 (Accuracy, FPR, FNR, MCC)...")

data = {'Model': df['Model']}

for metric in ['Accuracy', 'FPR', 'FNR', 'MCC']:
    means = []
    stds = []
    for val in df[metric]:
        mean, std, is_percentage = extract_mean_std(val)
        if mean is not None:
            # Convert percentage to 0-1 (except MCC which is already 0-1)
            if is_percentage and metric != 'MCC':
                mean = mean / 100
                std = std / 100
            means.append(mean)
            stds.append(std)
    
    data[f'MEAN_{metric}'] = means
    data[f'SD_{metric}'] = stds

sheet1 = pd.DataFrame(data)[['Model', 'MEAN_Accuracy', 'SD_Accuracy', 'MEAN_FPR', 'SD_FPR', 'MEAN_FNR', 'SD_FNR', 'MEAN_MCC', 'SD_MCC']]

# Reorder: CryptoGuard-MLP at last row
cryptoguard_row = sheet1[sheet1['Model'] == 'CryptoGuard-MLP']
other_rows = sheet1[sheet1['Model'] != 'CryptoGuard-MLP']
sheet1 = pd.concat([other_rows, cryptoguard_row], ignore_index=True)

# ============ SHEET 2: PRECISION, RECALL, F-MEASURE ============
print("[Sheet 2] Creating Chart_2 (Precision, Recall, F-measure)...")

data2 = {'Model': df['Model']}

for metric in ['Precision', 'Recall', 'F-measure']:
    means = []
    stds = []
    for val in df[metric]:
        mean, std, is_percentage = extract_mean_std(val)
        if mean is not None:
            # Convert percentage to 0-1
            if is_percentage:
                mean = mean / 100
                std = std / 100
            means.append(mean)
            stds.append(std)
    
    data2[f'MEAN_{metric}'] = means
    data2[f'SD_{metric}'] = stds

sheet2 = pd.DataFrame(data2)[['Model', 'MEAN_Precision', 'SD_Precision', 'MEAN_Recall', 'SD_Recall', 'MEAN_F-measure', 'SD_F-measure']]

# Reorder: CryptoGuard-MLP at last row
cryptoguard_row2 = sheet2[sheet2['Model'] == 'CryptoGuard-MLP']
other_rows2 = sheet2[sheet2['Model'] != 'CryptoGuard-MLP']
sheet2 = pd.concat([other_rows2, cryptoguard_row2], ignore_index=True)

# ============ SAVE TO EXCEL (2 SHEETS) ============
output_file = "./1. CryptoGuard-MLP_ComparativeAnalysis_13082026/1.3_CryptoGuard-MLP_ComparativeAnalysis_BarChartData_13082026.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    sheet1.to_excel(writer, sheet_name='Chart_1', index=False)
    sheet2.to_excel(writer, sheet_name='Chart_2', index=False)

print(f"\n✅ DONE!")
print(f"✓ Saved: {output_file}")

print("\n" + "="*80)
print("SHEET 1: Chart_1 (Accuracy, FPR, FNR, MCC)")
print("="*80)
print(sheet1.to_string(index=False))

print("\n" + "="*80)
print("SHEET 2: Chart_2 (Precision, Recall, F-measure)")
print("="*80)
print(sheet2.to_string(index=False))

print("\n" + "="*80)
print("BAR CHART DATA PREPARATION COMPLETE!")
print("="*80)
print("\n✓ Model Order:")
for idx, model in enumerate(sheet1['Model'], 1):
    if model == 'CryptoGuard-MLP':
        print(f"  {idx}. {model} ← LAST ROW")
    else:
        print(f"  {idx}. {model}")

In [3]:
import numpy as np

accuracy_values = [93.03, 92.58, 92.83, 93.70, 92.92]

mean = np.mean(accuracy_values)
std = np.std(accuracy_values)

print(f"{mean:.2f}% ± {std:.2f}%")
# Output: 93.01% ± 0.37%

93.01% ± 0.37%
